# Crime Diffusion Analysis in Bexley Using igraph

## Extension to: *Predicting Crime Hotspots in Bexley Using Metropolitan Police Data*

---

## Overview

This notebook extends the main Bexley hotspot project by introducing a **spatial network analysis** using the `igraph` library.

The core Random Forest model in the main project treats every LSOA as an independent observation — it has no concept of geography between areas. This notebook challenges that assumption by asking:

> **When an LSOA becomes a crime hotspot, do its geographic neighbours become hotspots in the following months?**

This phenomenon is known in criminology as **crime diffusion** or **near-repeat victimisation**, and it has real implications for how police and community safety teams should allocate resources.

---

## What this notebook does

1. Builds LSOA centroids from the existing crime dataset
2. Constructs a spatial network using `igraph` where LSOAs are nodes and edges connect nearby areas
3. Visualises the network on an interactive map
4. Measures crime diffusion rates compared to a baseline
5. Calculates a lift multiplier to quantify the strength of the diffusion effect
6. Identifies which LSOAs act as diffusion hubs — areas most likely to seed crime in their neighbours

---

## Prerequisites

This notebook assumes you have already run the main project notebook and that the following variables exist in your environment:

| Variable | Description |
|---|---|
| `df` | Incident-level Bexley street dataset with `LSOA name`, `Latitude`, `Longitude`, `Year`, `MonthNum` |
| `monthly_hotspots` | LSOA-month aggregated dataset with `LSOA name`, `Year`, `MonthNum`, `CrimeCount`, `Hotspot` |

If you are running this notebook standalone, run the data loading and feature engineering cells from the main notebook first, or re-run them in **Cell 2** of this notebook.

---
## Section 1 — Install and Import Libraries

The key new library here is `igraph`, a fast and well-documented graph analysis library. It is not included in the standard Colab environment so it needs to be installed first.

We also import `folium` for interactive maps, `scipy` for distance calculations, and the standard data science stack.

In [ ]:
# Install igraph (not included in Colab by default)
!pip install igraph -q
!pip install folium branca -q

# Graph analysis
import igraph as ig

# Data handling
import pandas as pd
import numpy as np

# Visualisation
import matplotlib.pyplot as plt
import matplotlib.cm as mplcm
import seaborn as sns
import folium
import branca.colormap as cm

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

print('igraph version:', ig.__version__)
print('All libraries imported successfully.')

---
## Section 2 — Load Data

**Skip this cell if you are running this notebook after the main project notebook** — `df` and `monthly_hotspots` will already exist in memory.

If you are starting fresh, this cell loads the dataset directly from GitHub using a raw URL. No file upload or manual download is needed — pandas reads it straight into memory over HTTP.

In [ ]:
# ── Only run this cell if df and monthly_hotspots are NOT already in memory ──
# Comment out this entire cell if continuing from the main notebook.

import requests
import io

# GitHub raw URL — converts the standard blob link to a direct download link
GITHUB_RAW_URL = (
    'https://raw.githubusercontent.com/callumhudson0/Generative-AI-Final-Project/main/'
    'combined_metropolitan_data_bexley_street_only.csv'
)

# ssl_verify=False is needed on macOS where Python's SSL certificates
# are not linked to the system keychain by default.
# This is safe here because we are reading from a known, trusted public URL.
print('Loading dataset from GitHub...')
response = requests.get(GITHUB_RAW_URL, verify=False)
response.raise_for_status()
df = pd.read_csv(io.StringIO(response.text))
print(f'Loaded successfully: {df.shape[0]:,} rows, {df.shape[1]} columns')

# Recreate time features
df['Month'] = pd.to_datetime(df['Month'])
df['Year'] = df['Month'].dt.year
df['MonthNum'] = df['Month'].dt.month

# Recreate the LSOA-month hotspot dataset
monthly_hotspots = df.groupby(['LSOA name', 'Year', 'MonthNum']).size().reset_index(name='CrimeCount')
hotspot_threshold = monthly_hotspots['CrimeCount'].quantile(0.75)
monthly_hotspots['Hotspot'] = (monthly_hotspots['CrimeCount'] >= hotspot_threshold).astype(int)

print(f'Dataset loaded: {df.shape[0]:,} incidents across {df["LSOA name"].nunique()} LSOAs')
print(f'Hotspot threshold (75th percentile): {hotspot_threshold}')
print(f'Monthly hotspot records: {len(monthly_hotspots):,}')

---
## Section 3 — Build LSOA Centroids

Before we can build a spatial network, we need a single geographic point for each LSOA.

We calculate the **centroid** of each LSOA by taking the mean latitude and longitude across all crime incidents recorded within it. This gives us a representative location for each area that we can use to measure distances between LSOAs.

This approach is an approximation — the true centroid of an LSOA boundary polygon would be slightly different — but it is accurate enough for a distance-based network at the borough level.

In [ ]:
# Calculate the mean lat/lon for each LSOA as its centroid
lsoa_centroids = (
    df.groupby('LSOA name')
      .agg(Lat=('Latitude', 'mean'), Lon=('Longitude', 'mean'))
      .reset_index()
)

# Also attach the total crime count for later use in maps and charts
lsoa_crime_totals = df.groupby('LSOA name').size().reset_index(name='TotalCrimes')
lsoa_centroids = lsoa_centroids.merge(lsoa_crime_totals, on='LSOA name', how='left')

print(f'Centroids built for {len(lsoa_centroids)} LSOAs')
print(f'Lat range: {lsoa_centroids["Lat"].min():.4f} to {lsoa_centroids["Lat"].max():.4f}')
print(f'Lon range: {lsoa_centroids["Lon"].min():.4f} to {lsoa_centroids["Lon"].max():.4f}')
lsoa_centroids.head()

---
## Section 4 — Build the Spatial Network with igraph

We now construct the spatial network. The logic is straightforward:

- Each **LSOA is a node** in the graph
- Two LSOAs are connected by an **edge** if their centroids are within a chosen distance threshold
- The edge **weight** is the distance in kilometres between the two centroids

### Choosing the distance threshold

The threshold of **1.2 km** is chosen because it roughly corresponds to the size of adjacent LSOA boundaries in a London borough. At this scale, connected LSOAs are genuinely neighbouring areas that share streets, not just areas in the same general part of the borough.

You can experiment with this value:
- **Smaller threshold (e.g. 0.8 km)** → sparser graph, only the closest neighbours connected
- **Larger threshold (e.g. 2.0 km)** → denser graph, more connections per LSOA

### Why use the Haversine formula?

Latitude and longitude are spherical coordinates, not flat grid coordinates. A naive Euclidean distance in degrees would slightly overestimate distances because the Earth is curved. The Haversine formula accounts for this curvature and returns accurate distances in kilometres.

In [ ]:
# ── Haversine distance function ──────────────────────────────────────────────
# Returns the great-circle distance in km between two lat/lon points

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371  # Earth's radius in km
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = (np.sin(dlat / 2) ** 2
         + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon / 2) ** 2)
    return R * 2 * np.arcsin(np.sqrt(a))


# ── Build the pairwise distance matrix ───────────────────────────────────────

DISTANCE_THRESHOLD_KM = 1.2  # Adjust this to change network density

lats = lsoa_centroids['Lat'].values
lons = lsoa_centroids['Lon'].values
n = len(lsoa_centroids)

distance_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        distance_matrix[i, j] = haversine_km(lats[i], lons[i], lats[j], lons[j])


# ── Identify edges (pairs of LSOAs within threshold) ─────────────────────────

edges = []
edge_weights = []
for i in range(n):
    for j in range(i + 1, n):   # i+1 avoids duplicates and self-loops
        dist = distance_matrix[i, j]
        if dist <= DISTANCE_THRESHOLD_KM:
            edges.append((i, j))
            edge_weights.append(round(dist, 3))


# ── Construct the igraph Graph object ────────────────────────────────────────

G = ig.Graph()
G.add_vertices(n)

# Attach LSOA attributes to each node
G.vs['name']         = lsoa_centroids['LSOA name'].tolist()
G.vs['lat']          = lats.tolist()
G.vs['lon']          = lons.tolist()
G.vs['total_crimes'] = lsoa_centroids['TotalCrimes'].tolist()

# Add edges and attach distance as edge weight
G.add_edges(edges)
G.es['distance_km'] = edge_weights


# ── Summary ───────────────────────────────────────────────────────────────────

degrees = G.degree()
print('=== Spatial Network Summary ===')
print(f'  Nodes (LSOAs):              {G.vcount()}')
print(f'  Edges (neighbour links):    {G.ecount()}')
print(f'  Distance threshold:         {DISTANCE_THRESHOLD_KM} km')
print(f'  Average neighbours per LSOA: {np.mean(degrees):.1f}')
print(f'  Max neighbours:             {max(degrees)}')
print(f'  Isolated LSOAs (0 neighbours): {sum(d == 0 for d in degrees)}')
print(f'  Is the graph connected?     {G.is_connected()}')

---
## Section 5 — Visualise the Network on an Interactive Map

This map shows the spatial network overlaid on Bexley. Each circle is an LSOA node:

- **Circle size** reflects the number of neighbours (network degree) — larger circles are more connected LSOAs
- **Circle colour** reflects total crime count — darker red means more crimes recorded
- **Grey lines** are the network edges connecting nearby LSOAs

This visual helps you sense-check the network before running the analysis. LSOAs in the denser central part of Bexley should have more connections; isolated LSOAs on the borough boundary may have fewer.

In [ ]:
# Build colour scale based on total crime count
colormap = cm.linear.YlOrRd_09.scale(
    lsoa_centroids['TotalCrimes'].min(),
    lsoa_centroids['TotalCrimes'].max()
)
colormap.caption = 'Total Crime Count per LSOA'

# Initialise map centred on Bexley
network_map = folium.Map(
    location=[lsoa_centroids['Lat'].mean(), lsoa_centroids['Lon'].mean()],
    zoom_start=12,
    tiles='CartoDB positron'
)
colormap.add_to(network_map)

# Draw edges first so nodes appear on top
for edge in G.es:
    src = lsoa_centroids.iloc[edge.source]
    tgt = lsoa_centroids.iloc[edge.target]
    folium.PolyLine(
        locations=[[src['Lat'], src['Lon']], [tgt['Lat'], tgt['Lon']]],
        color='#6b7280',
        weight=1.5,
        opacity=0.4
    ).add_to(network_map)

# Draw nodes — size scaled by degree, colour by crime count
for v in G.vs:
    row = lsoa_centroids.iloc[v.index]
    degree = G.degree(v.index)
    folium.CircleMarker(
        location=[row['Lat'], row['Lon']],
        radius=5 + degree,
        popup=(
            f"<b>{row['LSOA name']}</b><br>"
            f"Total crimes: {int(row['TotalCrimes'])}<br>"
            f"Network neighbours: {degree}"
        ),
        color=colormap(row['TotalCrimes']),
        fill=True,
        fill_color=colormap(row['TotalCrimes']),
        fill_opacity=0.85
    ).add_to(network_map)

network_map

---
## Section 6 — Measure Crime Diffusion

This is the core analytical step.

### How the measurement works

For every LSOA-month where a hotspot is recorded, we look at that LSOA's graph **neighbours** and ask: did those neighbours also become hotspots in the **following month (lag 1)** or **two months later (lag 2)**?

We compare this against a **baseline rate** — the overall probability that any LSOA becomes a hotspot in a given month, regardless of what its neighbours are doing.

$$\text{Lift} = \frac{\text{Diffusion Rate}}{\text{Baseline Rate}}$$

- A **lift > 1.0** means neighbours of hotspot LSOAs become hotspots more often than chance — evidence of diffusion
- A **lift = 1.0** means there is no spatial relationship — being next to a hotspot gives no extra information
- A **lift < 1.0** would suggest some kind of displacement effect (unusual and worth investigating)

### Why two lag windows?

Comparing lag 1 and lag 2 tells us something about the *speed* of diffusion:
- If lift is high at lag 1 but drops at lag 2, crime spreads quickly and fades fast
- If lift remains high at lag 2, the diffusion effect is more sustained

In [ ]:
# ── Prepare a fast lookup for hotspot status ──────────────────────────────────

hs = monthly_hotspots.copy().sort_values(['LSOA name', 'Year', 'MonthNum']).reset_index(drop=True)

# Create a sequential period index so we can do lag arithmetic easily
hs['Period'] = hs['Year'].astype(str) + '-' + hs['MonthNum'].astype(str).str.zfill(2)
all_periods = sorted(hs['Period'].unique())
period_to_idx = {p: i for i, p in enumerate(all_periods)}
hs['PeriodIdx'] = hs['Period'].map(period_to_idx)

# Dictionary: (LSOA name, PeriodIdx) -> Hotspot (0 or 1)
hotspot_lookup = hs.set_index(['LSOA name', 'PeriodIdx'])['Hotspot'].to_dict()

# Dictionary: LSOA name -> list of neighbour LSOA names (from igraph)
lsoa_names_in_graph = set(G.vs['name'])
neighbour_lookup = {
    v['name']: [G.vs[n]['name'] for n in G.neighbors(v.index)]
    for v in G.vs
}

print(f'Periods in dataset: {len(all_periods)} months')
print(f'Hotspot lookup entries: {len(hotspot_lookup):,}')


# ── Run diffusion measurement for lag 1 and lag 2 ────────────────────────────

results = []

for lag in [1, 2]:
    diffusion_cases  = 0   # neighbours of hotspots that became hotspots at T+lag
    diffusion_total  = 0   # total neighbour-month pairs observed after a hotspot
    baseline_cases   = 0   # LSOA-months where the LSOA became a hotspot at T+lag
    baseline_total   = 0   # all LSOA-months where T+lag data existed

    for _, row in hs.iterrows():
        lsoa       = row['LSOA name']
        period_idx = row['PeriodIdx']
        future_idx = period_idx + lag

        # Skip if the future period falls outside the dataset
        if future_idx >= len(all_periods):
            continue

        # ── Baseline: does this LSOA itself become a hotspot lag months later? ──
        future_status = hotspot_lookup.get((lsoa, future_idx), np.nan)
        if not np.isnan(future_status):
            baseline_total += 1
            baseline_cases += int(future_status)

        # ── Diffusion: if this LSOA IS a hotspot now, do its neighbours follow? ──
        if row['Hotspot'] == 1 and lsoa in neighbour_lookup:
            for neighbour in neighbour_lookup[lsoa]:
                neighbour_future = hotspot_lookup.get((neighbour, future_idx), np.nan)
                if not np.isnan(neighbour_future):
                    diffusion_total += 1
                    diffusion_cases += int(neighbour_future)

    diffusion_rate = diffusion_cases / diffusion_total if diffusion_total > 0 else 0
    baseline_rate  = baseline_cases  / baseline_total  if baseline_total  > 0 else 0
    lift           = diffusion_rate  / baseline_rate   if baseline_rate   > 0 else 0

    results.append({
        'Lag (months)':            lag,
        'Diffusion Rate':          round(diffusion_rate, 4),
        'Baseline Rate':           round(baseline_rate, 4),
        'Lift':                    round(lift, 3),
        'Diffusion observations':  diffusion_total,
        'Baseline observations':   baseline_total
    })

diffusion_results = pd.DataFrame(results)

print('=== Crime Diffusion Results ===')
print(diffusion_results.to_string(index=False))
print()
print('Plain-language interpretation:')
for _, row in diffusion_results.iterrows():
    direction = 'ABOVE' if row['Lift'] > 1.0 else 'AT or BELOW'
    print(f"  Lag {int(row['Lag (months)'])} month(s):  neighbours of hotspot LSOAs become hotspots "
          f"{row['Diffusion Rate']*100:.1f}% of the time  |  "
          f"baseline = {row['Baseline Rate']*100:.1f}%  |  "
          f"lift = {row['Lift']}x  ({direction} baseline)")

---
## Section 7 — Visualise the Diffusion Results

Two charts are produced:

1. **Diffusion Rate vs. Baseline Rate** — a grouped bar chart comparing how often neighbours of hotspot LSOAs become hotspots (red) versus the background rate for all LSOAs (grey), for each lag window

2. **Lift Multiplier** — a bar chart showing the lift score for each lag. The dashed line at 1.0 marks the boundary between no effect and a positive diffusion signal. A bar above this line and coloured red indicates meaningful spatial crime spread.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    'Crime Diffusion in Bexley: Do Hotspots Spread to Neighbouring LSOAs?',
    fontsize=13, fontweight='bold', y=1.03
)

labels = [f'Lag {int(r["Lag (months)"])} month' for _, r in diffusion_results.iterrows()]
x      = np.arange(len(labels))
width  = 0.35

# ── Chart 1: Diffusion Rate vs Baseline Rate ──────────────────────────────────
bars1 = axes[0].bar(
    x - width / 2,
    diffusion_results['Diffusion Rate'] * 100,
    width, label='Neighbours of hotspot LSOAs', color='#dc2626', alpha=0.85
)
bars2 = axes[0].bar(
    x + width / 2,
    diffusion_results['Baseline Rate'] * 100,
    width, label='All LSOAs (baseline)', color='#6b7280', alpha=0.7
)
axes[0].set_title('Hotspot Rate: Neighbours of Hotspots vs. All LSOAs')
axes[0].set_ylabel('% of observations that became a hotspot')
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels)
axes[0].legend()

max_rate = max(diffusion_results['Diffusion Rate'].max(), diffusion_results['Baseline Rate'].max())
axes[0].set_ylim(0, max_rate * 100 * 1.35)

for bar in list(bars1) + list(bars2):
    axes[0].annotate(
        f'{bar.get_height():.1f}%',
        xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
        xytext=(0, 4), textcoords='offset points', ha='center', fontsize=10
    )

# ── Chart 2: Lift Multiplier ──────────────────────────────────────────────────
lift_colors = [
    '#dc2626' if l > 1.2 else '#f97316' if l > 1.0 else '#6b7280'
    for l in diffusion_results['Lift']
]
bars3 = axes[1].bar(labels, diffusion_results['Lift'], color=lift_colors, alpha=0.85)
axes[1].axhline(y=1.0, color='black', linestyle='--', linewidth=1.5,
                label='No diffusion effect (lift = 1.0)')
axes[1].set_title('Diffusion Lift Multiplier by Lag Window')
axes[1].set_ylabel('Lift  (diffusion rate ÷ baseline rate)')
axes[1].legend()
axes[1].set_ylim(0, max(diffusion_results['Lift'].max() * 1.35, 1.6))

for bar in bars3:
    axes[1].annotate(
        f'{bar.get_height():.2f}x',
        xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
        xytext=(0, 5), textcoords='offset points',
        ha='center', fontsize=12, fontweight='bold'
    )

plt.tight_layout()
plt.show()

---
## Section 8 — Identify Crime Diffusion Hubs

Not all LSOAs are equally likely to seed crime in their neighbours. A **diffusion hub** is an LSOA that combines two properties:

1. **High hotspot frequency** — it is persistently a crime hotspot across many months
2. **High network degree** — it has many geographic neighbours through which crime could spread

We calculate a simple **hub score** as:

$$\text{Hub Score} = \text{Hotspot Rate} \times \text{Network Degree}$$

This is an intuitive measure: an LSOA that is frequently hot *and* well-connected poses the greatest diffusion risk to the surrounding neighbourhood.

LSOAs with a high hub score are candidates for prioritised attention in crime prevention planning — not just because they are high-crime themselves, but because they may be seeding elevated crime in the areas around them.

In [ ]:
# ── Calculate hotspot frequency per LSOA ─────────────────────────────────────

hotspot_freq = (
    monthly_hotspots
    .groupby('LSOA name')['Hotspot']
    .agg(HotspotMonths='sum', TotalMonths='count')
    .reset_index()
)
hotspot_freq['HotspotRate'] = (hotspot_freq['HotspotMonths'] / hotspot_freq['TotalMonths']).round(3)

# ── Attach network degree from igraph ────────────────────────────────────────

def get_degree(lsoa_name):
    matches = [v for v in G.vs if v['name'] == lsoa_name]
    return G.degree(matches[0].index) if matches else 0

hotspot_freq['Degree'] = hotspot_freq['LSOA name'].apply(get_degree)

# ── Calculate hub score ───────────────────────────────────────────────────────

hotspot_freq['HubScore'] = (hotspot_freq['HotspotRate'] * hotspot_freq['Degree']).round(3)

hub_ranking = hotspot_freq.sort_values('HubScore', ascending=False).head(10).reset_index(drop=True)

print('=== Top 10 Potential Crime Diffusion Hubs ===')
print(hub_ranking[['LSOA name', 'HotspotMonths', 'TotalMonths', 'HotspotRate', 'Degree', 'HubScore']]
      .to_string(index=False))


# ── Plot hub ranking ──────────────────────────────────────────────────────────

top10_plot = hub_ranking.sort_values('HubScore')  # ascending for horizontal bar
colors = plt.cm.Reds(np.linspace(0.35, 0.9, len(top10_plot)))

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.barh(top10_plot['LSOA name'], top10_plot['HubScore'], color=colors)

for bar, (_, row) in zip(bars, top10_plot.iterrows()):
    ax.text(
        bar.get_width() + 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"rate={row['HotspotRate']:.2f}, degree={int(row['Degree'])}",
        va='center', fontsize=9, color='#374151'
    )

ax.set_xlabel('Hub Score  (Hotspot Rate × Network Degree)', fontsize=11)
ax.set_title('Top 10 Potential Crime Diffusion Hubs in Bexley', fontsize=13, fontweight='bold')
ax.set_xlim(0, top10_plot['HubScore'].max() * 1.4)
plt.tight_layout()
plt.show()

---
## Section 9 — Map the Diffusion Hubs

This map overlays the hub score findings on the spatial network, making it easy to see which parts of Bexley contain the highest-risk diffusion hubs and how those areas connect to their neighbours.

- **Red circles** are the top 10 diffusion hub LSOAs
- **Blue circles** are all other LSOAs
- Circle size reflects hub score (for top 10) or network degree (for others)
- Click any node for its name, hub score and number of neighbours

In [ ]:
top_hub_names = set(hub_ranking['LSOA name'])
hub_score_lookup = hotspot_freq.set_index('LSOA name')['HubScore'].to_dict()
max_hub_score = hotspot_freq['HubScore'].max()

hub_map = folium.Map(
    location=[lsoa_centroids['Lat'].mean(), lsoa_centroids['Lon'].mean()],
    zoom_start=12,
    tiles='CartoDB positron'
)

# Draw edges
for edge in G.es:
    src = lsoa_centroids.iloc[edge.source]
    tgt = lsoa_centroids.iloc[edge.target]
    folium.PolyLine(
        locations=[[src['Lat'], src['Lon']], [tgt['Lat'], tgt['Lon']]],
        color='#9ca3af', weight=1.2, opacity=0.4
    ).add_to(hub_map)

# Draw nodes
for v in G.vs:
    row    = lsoa_centroids.iloc[v.index]
    name   = v['name']
    degree = G.degree(v.index)
    hub_score = hub_score_lookup.get(name, 0)
    is_hub    = name in top_hub_names

    folium.CircleMarker(
        location=[row['Lat'], row['Lon']],
        radius=8 + (hub_score / max_hub_score * 14) if is_hub else 5 + degree * 0.5,
        popup=(
            f"<b>{name}</b><br>"
            f"{'⚠️ Top 10 Diffusion Hub<br>' if is_hub else ''}"
            f"Hub Score: {hub_score:.3f}<br>"
            f"Network Neighbours: {degree}"
        ),
        color='#dc2626' if is_hub else '#3b82f6',
        fill=True,
        fill_color='#dc2626' if is_hub else '#93c5fd',
        fill_opacity=0.85 if is_hub else 0.6
    ).add_to(hub_map)

# Add a simple legend
legend_html = '''
<div style="position: fixed; bottom: 40px; left: 40px; z-index: 1000;
            background: white; padding: 12px 16px; border-radius: 8px;
            border: 1px solid #d1d5db; font-family: sans-serif; font-size: 13px;">
  <b>Legend</b><br>
  <span style="color:#dc2626;">●</span> Top 10 diffusion hub<br>
  <span style="color:#3b82f6;">●</span> Other LSOA<br>
  <span style="color:#9ca3af;">—</span> Neighbour link (&lt;1.2 km)
</div>
'''
hub_map.get_root().html.add_child(folium.Element(legend_html))

hub_map

---
## Section 10 — Summary and Interpretation

This section summarises the findings of the diffusion analysis and explains how they relate to the main project.

### What the results tell us

**1. Diffusion rate vs. baseline**
If the diffusion rate at lag 1 or lag 2 is meaningfully higher than the baseline rate, this is direct evidence that crime hotspots in Bexley are not spatially independent. Being a neighbour of a hotspot LSOA makes an area more vulnerable than a random LSOA in the same borough.

**2. Lift multiplier**
The lift score quantifies how much more likely a neighbour is to become a hotspot compared to any LSOA. A lift of 1.5x means neighbours of hotspot LSOAs are 50% more likely than average to become hotspots themselves in the following month.

**3. Lag comparison**
Comparing lag 1 and lag 2 reveals the *speed* of the diffusion effect:
- If lift is high at lag 1 but lower at lag 2 → crime spreads fast but the effect decays quickly
- If lift remains high at lag 2 → the spatial pressure on neighbouring areas persists over multiple months

**4. Diffusion hubs**
The hub score identifies LSOAs that combine persistent hotspot behaviour with high network connectivity. These areas are not only consistently high-crime themselves — they are the most likely sources of spatial crime spread. In a real planning context, these are the areas where early intervention could have the widest neighbourhood-level impact.

---

### Linking back to the main project

The Random Forest model in the main notebook treats each LSOA as an independent unit. The diffusion findings suggest this is an oversimplification — spatial context matters. A natural next step would be to enrich the Random Forest's feature set with **lagged neighbour hotspot status** (e.g. *'was at least one neighbour a hotspot last month?'*) and test whether this improves predictive performance.

This is one example of how graph-based analysis and machine learning can complement each other in a spatial crime analytics project.

In [ ]:
# Print a clean final summary table

print('=' * 65)
print('  BEXLEY CRIME DIFFUSION ANALYSIS — FINAL RESULTS SUMMARY')
print('=' * 65)
print(f'  Network: {G.vcount()} LSOAs, {G.ecount()} neighbour links (<{DISTANCE_THRESHOLD_KM} km)')
print(f'  Time period: {all_periods[0]} to {all_periods[-1]}  ({len(all_periods)} months)')
print()
print('  Diffusion results:')
for _, row in diffusion_results.iterrows():
    sig = '✓ DIFFUSION DETECTED' if row['Lift'] > 1.1 else '– No strong effect'
    print(f"    Lag {int(row['Lag (months)'])}:  "
          f"diffusion={row['Diffusion Rate']*100:.1f}%  "
          f"baseline={row['Baseline Rate']*100:.1f}%  "
          f"lift={row['Lift']}x  {sig}")
print()
print('  Top 3 diffusion hubs:')
for i, (_, row) in enumerate(hub_ranking.head(3).iterrows(), 1):
    print(f"    {i}. {row['LSOA name']}  "
          f"(hotspot {row['HotspotMonths']}/{row['TotalMonths']} months, "
          f"{row['Degree']} neighbours, hub score={row['HubScore']})")
print('=' * 65)